# $\xi(L)$ at fixed $\chi$ and $\nu$

Localization length from the exit-intensity profile, same estimator as the phase diagrams
(`20251211_Phase_Diagrams*.ipynb`): each sample's exit profile is normalized to its maximum,
profiles are averaged over the 5 samples, and $\xi/a = 1/\mathrm{IPR}$ with
$\mathrm{IPR} = \int I^2\,dx \,/\, (\int I\,dx)^2$.

Data: `./data/20251212_intensities_exit.h5`, shape `(chi, size, sample, x, freq)`; $L$ = `size`$\cdot a$.
The $\chi=0$ slot holds a single sample (samples 1–4 are empty), so it is not ensemble-averaged.

In [ ]:
%matplotlib inline
import numpy as np
import h5py
import scipy.integrate
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ipywidgets as widgets

plt.rc('font', family='Arial', size=12)
C_0 = 299792458e6  # um/s, so f/C_0 = a/lambda with a = 1 um

In [ ]:
data_path = "./data/20251212_intensities_exit.h5"

def ipr(intensity, x):
    # intensity: (..., x, freq) -> (..., freq)
    return (scipy.integrate.simpson(intensity**2, x=x, axis=-2)
            / scipy.integrate.simpson(intensity, x=x, axis=-2)**2)

with h5py.File(data_path, "r") as h5f:
    chis = h5f["chi"][:]
    sizes = h5f["size"][:]
    x = h5f["x"][:]
    nu = h5f["f"][:] / C_0
    n_chi, n_size, n_sample = h5f["data_field_intensities_2"].shape[:3]

    ipr_avg = np.zeros((n_chi, n_size, nu.size))               # IPR = a/xi of sample-averaged profile
    ipr_samples = np.zeros((n_chi, n_size, n_sample, nu.size))  # IPR of each sample
    for i in range(n_chi):
        block = h5f["data_field_intensities_2"][i]            # (size, sample, x, freq)
        peak = np.max(block, axis=2, keepdims=True)
        with np.errstate(invalid="ignore", divide="ignore"):
            block = np.where(peak > 0, block / peak, np.nan)  # empty samples -> NaN (chi=0 has only sample 0)
            ipr_avg[i] = ipr(np.nanmean(block, axis=1), x)
            ipr_samples[i] = ipr(block, x)

print("chi:", chis)
print("L/a:", sizes)
print(f"nu: {nu.min():.3f} - {nu.max():.3f} a/lambda, {nu.size} points")

In [ ]:
def moving_average(arr, w):
    # 'valid' moving average along the last axis, matching the phase-diagram smoothing
    if w <= 1:
        return arr
    kernel = np.ones(w) / w
    return np.apply_along_axis(lambda v: np.convolve(v, kernel, 'valid'), -1, arr)

chi_order = np.argsort(chis)

def plot_xi_vs_L(chi_idx, nu_target, window, show_samples, log_axes):
    nu_s = moving_average(nu, window)
    k = np.argmin(np.abs(nu_s - nu_target))
    # smooth a/xi, then invert (same order as the phase diagrams)
    xi = 1 / moving_average(ipr_avg[chi_idx], window)[:, k]
    xi_s = 1 / moving_average(ipr_samples[chi_idx], window)[:, :, k]

    fig, ax = plt.subplots(figsize=(6, 4), dpi=110, constrained_layout=True)
    if show_samples:
        for s in range(xi_s.shape[1]):
            ax.plot(sizes, xi_s[:, s], 'o', ms=3, color='gray', alpha=0.4,
                    label='single samples' if s == 0 else None)
    ax.plot(sizes, xi, 'o-', color='#0f084b', label='sample-averaged profile')
    ax.set_xlabel(r"$L/a$")
    ax.set_ylabel(r"$\xi/a$")
    if log_axes:
        ax.set_xscale('log')
        ax.set_yscale('log')
        for axis in (ax.xaxis, ax.yaxis):
            axis.set_major_locator(ticker.LogLocator(base=10, subs=(1, 2, 5)))
            axis.set_major_formatter(ticker.FormatStrFormatter('%g'))
            axis.set_minor_formatter(ticker.NullFormatter())
    ax.set_title(rf"$\chi={chis[chi_idx]:.2f}$,  $\nu={nu_s[k]:.4f}\,a/\lambda$"
                #  + (f"  (window {window})" if window > 1 else "")
                 )
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.show()

dnu = nu[1] - nu[0]
widgets.interact(
    plot_xi_vs_L,
    chi_idx=widgets.Dropdown(options=[(f"{chis[i]:.2f}", int(i)) for i in chi_order],
                             value=int(np.argmin(np.abs(chis - 0.37))), description="χ"),
    nu_target=widgets.FloatSlider(value=0.30, min=nu.min(), max=nu.max(), step=dnu,
                                  readout_format=".4f", description="ν (a/λ)",
                                  continuous_update=False,
                                  layout=widgets.Layout(width="500px")),
    window=widgets.IntSlider(value=5, min=1, max=15, description="ν smoothing"),
    show_samples=widgets.Checkbox(value=False, description="show single samples"),
    log_axes=widgets.Checkbox(value=False, description="log-log"),
);